In [ ]:
import polars as pl

In [2]:
# Define pathings
data_path = '../01_data/'
out_path = '../03_output/'

In [3]:
#Load data
pronouns_df = pl.read_csv(data_path + 'das_tfg_pronoun_study.csv', separator= ',' )

pronouns_df.head()

Left,KWIC,Right
str,str,str
"""<s> NARRATOR Yes, indeed. </s>…","""they""","""are locked away, to await the …"
"""<s> NARRATOR Yes, indeed. </s>…","""your""","""fate. </s><s> Only, in the anc…"
"""<s> NARRATOR Yes, indeed. </s>…","""it""","""is stated, that one day an Und…"
"""brands the Undead. </s><s> And…","""it""","""not so that thou art new. </s>…"
"""the Undead. </s><s> And in thi…","""thou""","""art new. </s><s> Thou fared we…"


In [4]:
#Load Character master table
char_master_df = pl.read_csv(data_path + 'das_char_master.csv')\
                   .with_columns(pl.col('Character').str.replace(',','').alias('Character'))

char_master_df.head()

Character,Class,Age
str,str,str
"""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ANASTACIA OF ASTORA""","""Low""","""Young"""
"""ANDRE OF ASTORA""","""Low""","""Old"""
"""BIG HAТ LOGAN""","""Low""","""Old"""
"""BLACKSMITH VAMOS""","""Low""","""Old"""


In [5]:
# # Function to extract the name of the speaking character from the left context.
# def name_finder(text, punctuation = [',','.',';',':','!','?']):
#     # Get rid of half sentences in the left context and from the end of sentence marker.
#     paragraphs = text.split('<s>')
#     paragraphs_full = ''.join(paragraphs[1:]).split('<s/>')
#     text = ' '.join(paragraphs_full)
#     # Remove punctuation from text.
#     for char in punctuation:
#         text = text.replace(char, '')
#     words = text.split(' ')
#     caps = []
#     # Search for words fully in uppercase that are longer than a single letter.
#     for word in words:
#         if (word == word.upper()) & (len(word) > 1):
#             caps.append(word)
#     caps = ' '.join(caps)
#     # If there are none return a null value to make forward filling easier.
#     if caps == '':
#         caps = None
#     return caps

In [6]:
# Function to extract the name of the speaking character from the left context.
def name_finder(text, punctuation = [',','.',';',':','!','?']):
    # Get rid of half sentences in the left context and from the end of sentence marker.
    paragraphs = text.split('<s>')
    paragraphs_full = ''.join(paragraphs[1:]).split('</s>')
    text = ' '.join(paragraphs_full)
    # Remove punctuation from text.
    for char in punctuation:
        text = text.replace(char, '')
    text = text.replace('\n', ' ')
    words = text.split(' ')
    caps = ['|']
    previous_was_caps = False
    # Search for words fully in uppercase that are longer than a single letter.
    for word in words:
        if (word == word.upper()) & (len(word) > 1):
            caps.append(word)
            previous_was_caps = True
            continue
        # In case there are more than one name in the captured context, select the last to appear.
        if previous_was_caps:
            caps.append('|')
            previous_was_caps = False
    # print(caps)
    if caps[-1] == '|':
        caps = caps[:-1]
    caps = ' '.join(caps)
    caps = caps.split('|')[-1].strip()
    # If there are none return a null value to make forward filling easier.
    if caps == '':
        caps = None
    return caps

In [7]:
# Extract the speaking character for each row, turn all KWIC lowercase to avoid redundant values
pronouns_df = pronouns_df.with_columns(pl.col('Left').map_elements(name_finder, return_dtype=pl.Utf8).alias('Character'))\
                         .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                         .with_columns(pl.col('Character').forward_fill().alias('Character'))\
                         .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>','\n'),
                                       pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>','\n'))\

pronouns_df
                         

Left,KWIC,Right,Character
str,str,str,str
""" NARRATOR Yes, indeed. The D…","""they""","""are locked away, to await the …","""NARRATOR"""
""" NARRATOR Yes, indeed. The D…","""your""","""fate. Only, in the ancient l…","""NARRATOR"""
""" NARRATOR Yes, indeed. The D…","""it""","""is stated, that one day an Und…","""NARRATOR"""
"""brands the Undead. And in th…","""it""","""not so that thou art new. Th…","""ALVINA OF THE DARKROOT WOOD"""
"""the Undead. And in this land…","""thou""","""art new. Thou fared well to …","""ALVINA OF THE DARKROOT WOOD"""
…,…,…,…
""" You''re a persistent one, are…","""we""","""meet again. Vereor Nox. Eg…","""VINCE OF THOROLUND"""
"""with your kind. But there''s…","""you""","""? I cannot overlook a threat…","""VINCE OF THOROLUND"""
"""your kind. But there''s not …","""i""","""cannot overlook a threat to M'…","""VINCE OF THOROLUND"""


In [8]:
#Add character class and age information from master table
pronouns_df = pronouns_df.join(char_master_df, on='Character', how = 'left')

pronouns_df

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str
""" NARRATOR Yes, indeed. The D…","""they""","""are locked away, to await the …","""NARRATOR""",null,null
""" NARRATOR Yes, indeed. The D…","""your""","""fate. Only, in the ancient l…","""NARRATOR""",null,null
""" NARRATOR Yes, indeed. The D…","""it""","""is stated, that one day an Und…","""NARRATOR""",null,null
"""brands the Undead. And in th…","""it""","""not so that thou art new. Th…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""the Undead. And in this land…","""thou""","""art new. Thou fared well to …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
…,…,…,…,…,…
""" You''re a persistent one, are…","""we""","""meet again. Vereor Nox. Eg…","""VINCE OF THOROLUND""","""Low""","""Young"""
"""with your kind. But there''s…","""you""","""? I cannot overlook a threat…","""VINCE OF THOROLUND""","""Low""","""Young"""
"""your kind. But there''s not …","""i""","""cannot overlook a threat to M'…","""VINCE OF THOROLUND""","""Low""","""Young"""


In [9]:
# Sanity check for extracted characters
pronouns_df.to_pandas()['Character'].unique()

array(['NARRATOR', 'ALVINA OF THE DARKROOT WOOD', 'ANASTACIA OF ASTORA',
       'ANDRE OF ASTORA', 'BIG HAТ LOGAN', 'BLACKSMITH VAMOS',
       'CRESTFALLEN MERCHANT', 'CRESTFALLEN WARRIOR',
       'CROSSBREED PRISCILLA', 'DARK SUN GWYNDOLIN', 'DARKMOON KNIGHTESS',
       'DARKSTALKER KAATHЕ', 'DOMHNALL OF ZENA', 'DUSK OF OOLACILE',
       'EINGYI OF THE GREAT SWAMP', 'ELIZABETH KEEPER OF THE SANCTUARY',
       'GIANT BLACKSMITH', 'GRIGGS OF VINHEIM',
       'GWYNEVERE PRINCESS OF SUNLIGHT', 'HAWKEYE GOUGН',
       'INGWARD KEEPER OF THE SEAL', 'KINGSEEKER FRAMPТ',
       'LAURENTIUS OF THЕ GREAT SWAMP', 'LAUTREC OF CARIM',
       "LORD''S BLADE CIARAN", 'MARVELOUS CHESTER', 'OSCAR OF ASTORA',
       'OSWALD OF CARIM', 'PETRUS OF THOROLUND', 'QUELANA OF IZALITH',
       'RHEA OF THOROLUND', 'RICKERT OF VINHEIM', 'SHIVA OF THE EAST',
       'SIEGLINDE OF CATARINA', 'SIEGMEYER OF CATARINA', 'HAWK GIRL',
       'SOLAIRE OF ASTORA', 'THE FAIR LADY', 'TRUSTY PATCHES',
       'UNDEAD MERCHANT

In [10]:
# Save enriched dataframe into a csv file.
pronouns_df.write_csv(out_path + 'pronoun_analysis.csv', separator= ';')